# Baseline Balanced NSL-KDD — DNN, Without Feature Augmentation and Without Optuna

## 1. Imports, dataset path, and reproducibility


In [4]:
# ============================================================
# Imports and fixed experiment seed
# ============================================================
import gc
import json
import os
import platform
import random
import time
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, GRU, Input, LSTM
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam


# ============================================================
# Pre-generated NSL-KDD augmented partitions
# ============================================================
AUGMENTED_DATA_DIRECTORY = (
    Path.cwd()
    / "/kaggle/input/datasets/sofonyasz/nsl-kdd-manuscript"
)

TRAIN_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "NSL_KDD_Augmented_Train.csv"
)

VALIDATION_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "NSL_KDD_Augmented_Validation.csv"
)

TEST_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "NSL_KDD_Augmented_Test.csv"
)

# Retained for compatibility with the existing experiment metadata.
DATASET_PATH = AUGMENTED_DATA_DIRECTORY


SEED = 42
RANDOM_STATE = SEED

os.environ[
    "PYTHONHASHSEED"
] = str(SEED)

random.seed(
    SEED
)

np.random.seed(
    SEED
)

tf.keras.utils.set_random_seed(
    SEED
)

print(
    "Fixed experiment seed:",
    SEED,
)

print(
    "Training file:",
    TRAIN_FILE,
)

print(
    "Validation file:",
    VALIDATION_FILE,
)

print(
    "Test file:",
    TEST_FILE,
)


Fixed experiment seed: 42
Training file: /kaggle/input/datasets/sofonyasz/nsl-kdd-manuscript/NSL_KDD_Augmented_Train.csv
Validation file: /kaggle/input/datasets/sofonyasz/nsl-kdd-manuscript/NSL_KDD_Augmented_Validation.csv
Test file: /kaggle/input/datasets/sofonyasz/nsl-kdd-manuscript/NSL_KDD_Augmented_Test.csv


In [5]:
# ============================================================
# Load the pre-generated NSL-KDD augmented partitions
# ============================================================
for file_path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE,
]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Required augmented partition not found: {file_path.resolve()}"
        )

train_data = pd.read_csv(
    TRAIN_FILE
)

validation_data = pd.read_csv(
    VALIDATION_FILE
)

test_data = pd.read_csv(
    TEST_FILE
)


# ============================================================
# Basic partition validation
# ============================================================
if train_data.empty:
    raise ValueError(
        "The training dataset is empty."
    )

if validation_data.empty:
    raise ValueError(
        "The validation dataset is empty."
    )

if test_data.empty:
    raise ValueError(
        "The test dataset is empty."
    )


LABEL_COLUMN = "category"

for dataset_name, dataset in {
    "Training": train_data,
    "Validation": validation_data,
    "Test": test_data,
}.items():
    if LABEL_COLUMN not in dataset.columns:
        raise KeyError(
            f"'{LABEL_COLUMN}' was not found in the "
            f"{dataset_name} partition."
        )

    if dataset[
        LABEL_COLUMN
    ].isna().any():
        raise ValueError(
            f"'{LABEL_COLUMN}' contains missing values "
            f"in the {dataset_name} partition."
        )


print(
    f"Training data:   {train_data.shape}"
)

print(
    f"Validation data: {validation_data.shape}"
)

print(
    f"Test data:       {test_data.shape}"
)

display(
    train_data.head()
)


Training data:   (104258, 44)
Validation data: (14826, 44)
Test data:       (29433, 44)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,IF_Anomaly_Score,AE_Anomaly_Score,SUOD_Anomaly_Score,category
0,0.0,1.0,11.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,1.0,1.0,0.00,0.0,-0.409113,0.008093,-0.155286,DoS
1,0.0,1.0,49.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,1.00,1.0,-0.455302,0.095445,0.259301,DoS
2,0.0,1.0,20.0,9.0,172.0,0.0,0.0,0.0,0.0,0.0,...,0.18,0.00,0.4,0.0,0.02,0.0,-0.421444,0.086396,-0.270910,Normal
3,0.0,1.0,24.0,9.0,233.0,547.0,0.0,0.0,0.0,0.0,...,0.03,0.03,0.0,0.0,0.00,0.0,-0.361821,0.009765,-0.623136,Normal
4,0.0,1.0,24.0,9.0,330.0,453.0,0.0,0.0,0.0,0.0,...,0.01,0.02,0.0,0.0,0.00,0.0,-0.347595,0.004767,-0.797096,Normal


## 2. Fixed stratified NSL-KDD train/validation/test split


In [6]:
# ============================================================
# Feature-target separation and validation
# ============================================================
anomaly_score_columns = [
    "IF_Anomaly_Score",
    "AE_Anomaly_Score",
    "SUOD_Anomaly_Score",
]

for dataset_name, dataset in {
    "Training": train_data,
    "Validation": validation_data,
    "Test": test_data,
}.items():
    missing_anomaly_features = [
        feature
        for feature in anomaly_score_columns
        if feature not in dataset.columns
    ]

    if missing_anomaly_features:
        raise KeyError(
            f"{dataset_name} partition is missing anomaly-score features: "
            f"{missing_anomaly_features}"
        )


# ============================================================
# Remove anomaly-score features for the No-FA experiment
# ============================================================
train_data = train_data.drop(
    columns=anomaly_score_columns
)

validation_data = validation_data.drop(
    columns=anomaly_score_columns
)

test_data = test_data.drop(
    columns=anomaly_score_columns
)


feature_columns = [
    column
    for column in train_data.columns
    if column != LABEL_COLUMN
]


for dataset_name, dataset in {
    "Validation": validation_data,
    "Test": test_data,
}.items():
    current_features = [
        column
        for column in dataset.columns
        if column != LABEL_COLUMN
    ]

    if current_features != feature_columns:
        raise ValueError(
            f"{dataset_name} feature columns/order do not match training."
        )


X_train_raw = train_data[
    feature_columns
].copy()

X_val_raw = validation_data[
    feature_columns
].copy()

X_test_raw = test_data[
    feature_columns
].copy()


y_train_raw = (
    train_data[
        LABEL_COLUMN
    ]
    .astype(str)
    .str.strip()
)

y_val_raw = (
    validation_data[
        LABEL_COLUMN
    ]
    .astype(str)
    .str.strip()
)

y_test_raw = (
    test_data[
        LABEL_COLUMN
    ]
    .astype(str)
    .str.strip()
)


for dataset_name, labels in {
    "Training": y_train_raw,
    "Validation": y_val_raw,
    "Test": y_test_raw,
}.items():
    if labels.eq("").any():
        raise ValueError(
            f"{dataset_name} target labels contain empty values."
        )


for dataset_name, feature_data in {
    "Training": X_train_raw,
    "Validation": X_val_raw,
    "Test": X_test_raw,
}.items():
    non_numeric_columns = (
        feature_data
        .select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )

    if non_numeric_columns:
        raise TypeError(
            f"{dataset_name} contains non-numeric feature columns: "
            f"{non_numeric_columns}"
        )

    if not np.isfinite(
        feature_data.to_numpy(
            dtype=np.float64
        )
    ).all():
        raise ValueError(
            f"{dataset_name} features contain missing or infinite values."
        )


# Preserve row positions within each pre-generated partition so that
# the existing artifact-saving structure remains unchanged.
train_indices = (
    X_train_raw
    .index
    .to_numpy(
        copy=True
    )
)

validation_indices = (
    X_val_raw
    .index
    .to_numpy(
        copy=True
    )
)

test_indices = (
    X_test_raw
    .index
    .to_numpy(
        copy=True
    )
)


X_train_raw = (
    X_train_raw
    .reset_index(
        drop=True
    )
)

X_val_raw = (
    X_val_raw
    .reset_index(
        drop=True
    )
)

X_test_raw = (
    X_test_raw
    .reset_index(
        drop=True
    )
)

y_train_raw = (
    y_train_raw
    .reset_index(
        drop=True
    )
)

y_val_raw = (
    y_val_raw
    .reset_index(
        drop=True
    )
)

y_test_raw = (
    y_test_raw
    .reset_index(
        drop=True
    )
)


print(
    "Original training shape:",
    X_train_raw.shape,
)

print(
    "Validation shape:",
    X_val_raw.shape,
)

print(
    "Test shape:",
    X_test_raw.shape,
)


del train_data, validation_data, test_data


Original training shape: (104258, 40)
Validation shape: (14826, 40)
Test shape: (29433, 40)


In [7]:
# ============================================================
# Fit the label encoder on training labels only
# ============================================================
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    y_train_raw
)

training_classes = set(
    label_encoder.classes_
)

unknown_validation_classes = (
    set(
        y_val_raw.unique()
    )
    - training_classes
)

unknown_test_classes = (
    set(
        y_test_raw.unique()
    )
    - training_classes
)

if unknown_validation_classes:
    raise ValueError(
        "Validation data contain classes absent from training: "
        f"{sorted(unknown_validation_classes)}"
    )

if unknown_test_classes:
    raise ValueError(
        "Test data contain classes absent from training: "
        f"{sorted(unknown_test_classes)}"
    )


y_val = label_encoder.transform(
    y_val_raw
)

y_test = label_encoder.transform(
    y_test_raw
)

number_of_classes = len(
    label_encoder.classes_
)


# ============================================================
# Validate expected five NSL-KDD categories
# ============================================================
expected_nsl_kdd_classes = {
    "normal",
    "dos",
    "probe",
    "r2l",
    "u2r",
}

observed_nsl_kdd_classes = {
    str(
        class_name
    )
    .strip()
    .lower()
    for class_name in label_encoder.classes_
}

if (
    number_of_classes != 5
    or observed_nsl_kdd_classes
    != expected_nsl_kdd_classes
):
    raise ValueError(
        "Expected the five NSL-KDD categories "
        "Normal, DoS, Probe, R2L, and U2R; observed "
        f"{label_encoder.classes_.tolist()}."
    )


print(
    "Multiclass label mapping:"
)

for (
    class_code,
    class_name,
) in enumerate(
    label_encoder.classes_
):
    print(
        f"{class_code}: "
        f"{class_name}"
    )

print(
    "Number of classes:",
    number_of_classes,
)

print(
    "Training distribution:",
    Counter(
        map(
            int,
            y_train,
        )
    ),
)

print(
    "Validation distribution:",
    Counter(
        map(
            int,
            y_val,
        )
    ),
)

print(
    "Test distribution:",
    Counter(
        map(
            int,
            y_test,
        )
    ),
)


Multiclass label mapping:
0: DoS
1: Normal
2: Probe
3: R2L
4: U2R
Number of classes: 5
Training distribution: Counter({1: 54091, 0: 37477, 2: 9882, 3: 2724, 4: 84})
Validation distribution: Counter({1: 7692, 0: 5330, 2: 1405, 3: 387, 4: 12})
Test distribution: Counter({1: 15271, 0: 10580, 2: 2790, 3: 769, 4: 23})


## 3. U2R-only training SMOTE and standardization


In [8]:
# ============================================================
# U2R-only training-set SMOTE
# ============================================================
# Confirm that U2R is encoded as class 4.
u2r_class_codes = [
    class_code
    for (
        class_code,
        class_name,
    ) in enumerate(
        label_encoder.classes_
    )
    if (
        str(
            class_name
        )
        .strip()
        .lower()
        == "u2r"
    )
]

if len(
    u2r_class_codes
) != 1:
    raise ValueError(
        "Exactly one U2R class was expected, but the "
        "encoded mapping was: "
        f"{label_encoder.classes_.tolist()}"
    )

U2R_CLASS_CODE = int(
    u2r_class_codes[
        0
    ]
)

if U2R_CLASS_CODE != 4:
    raise ValueError(
        "The NSL-KDD balancing protocol expects U2R "
        "to be encoded as class 4. "
        f"Observed U2R code: {U2R_CLASS_CODE}"
    )


# Define the desired number of instances per class after
# resampling: raise U2R from 84 to 2,704 instances.
target_class_counts = {
    4: 84 + 2620
}


original_training_counts = Counter(
    map(
        int,
        y_train,
    )
)

print(
    "Training distribution before SMOTE:",
    original_training_counts,
)


u2r_training_count = int(
    original_training_counts[
        U2R_CLASS_CODE
    ]
)

if u2r_training_count != 84:
    raise ValueError(
        "The fixed NSL-KDD training split is expected "
        "to contain 84 U2R instances before SMOTE, "
        f"but {u2r_training_count} were found. "
        "Check the dataset and split configuration."
    )

if (
    target_class_counts[
        U2R_CLASS_CODE
    ]
    <= u2r_training_count
):
    raise ValueError(
        "The U2R SMOTE target must be greater than "
        "the original U2R training count."
    )


smote = SMOTE(
    sampling_strategy=target_class_counts,
    random_state=42,
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_raw,
    y_train,
)


print(
    "Training distribution after SMOTE:",
    Counter(
        map(
            int,
            y_train_resampled,
        )
    ),
)

print(
    "U2R target count:",
    target_class_counts[
        U2R_CLASS_CODE
    ],
)

print(
    "Resampled training shape:",
    X_train_resampled.shape,
)


Training distribution before SMOTE: Counter({1: 54091, 0: 37477, 2: 9882, 3: 2724, 4: 84})
Training distribution after SMOTE: Counter({1: 54091, 0: 37477, 2: 9882, 3: 2724, 4: 2704})
U2R target count: 2704
Resampled training shape: (106878, 40)


In [9]:
# ============================================================
# Fit StandardScaler on the SMOTE-resampled training set only.
# ============================================================
classifier_scaler = StandardScaler()

X_train_scaled = classifier_scaler.fit_transform(
    X_train_resampled
).astype(np.float32)

X_val_scaled = classifier_scaler.transform(
    X_val_raw
).astype(np.float32)

X_test_scaled = classifier_scaler.transform(
    X_test_raw
).astype(np.float32)

for partition_name, values in {
    "resampled training": X_train_scaled,
    "validation": X_val_scaled,
    "test": X_test_scaled,
}.items():
    if not np.isfinite(values).all():
        raise ValueError(f"Scaled {partition_name} contains invalid values.")

print(f"Resampled training mean after scaling: {X_train_scaled.mean():.6f}")
print(f"Resampled training standard deviation after scaling: {X_train_scaled.std():.6f}")
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)


Resampled training mean after scaling: 0.000000
Resampled training standard deviation after scaling: 1.000000
Scaled training shape: (106878, 40)
Scaled validation shape: (14826, 40)
Scaled test shape: (29433, 40)


## 4. Fixed baseline configuration — no Optuna


In [10]:
# ============================================================
# Fixed baseline settings from the uploaded notebook.
# No Optuna and no pre-stored optimized hyperparameters.
# ============================================================
RUN_SEEDS = [42, 43, 44, 45, 46]
N_FINAL_RUNS = len(RUN_SEEDS)

N_WARMUP_RUNS = 3
N_BENCHMARK_RUNS = 5

BASELINE_HYPERPARAMETERS = {
    "hidden_units_1": 64,
    "hidden_units_2": 32,
    "dropout_rate": 0.2,
    "learning_rate": 0.001,
    "l2_regularization": 1e-4,
    "epochs": 100,
    "batch_size": 64,
    "shuffle": True,
    "early_stopping": {
        "monitor": "val_loss",
        "patience": 10,
        "min_delta": 0.0,
        "restore_best_weights": True,
    },
}

print("Final training seeds:", RUN_SEEDS)
print("Fixed baseline hyperparameters:")
print(json.dumps(BASELINE_HYPERPARAMETERS, indent=2))


Final training seeds: [42, 43, 44, 45, 46]
Fixed baseline hyperparameters:
{
  "hidden_units_1": 64,
  "hidden_units_2": 32,
  "dropout_rate": 0.2,
  "learning_rate": 0.001,
  "l2_regularization": 0.0001,
  "epochs": 100,
  "batch_size": 64,
  "shuffle": true,
  "early_stopping": {
    "monitor": "val_loss",
    "patience": 10,
    "min_delta": 0.0,
    "restore_best_weights": true
  }
}


## 5. Evaluation utilities


In [11]:
# ============================================================
# Reproducibility, metrics, timing, and output utilities
# ============================================================
def set_final_run_seed(seed):
    """Reset only model-training random states for one independent run."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except (AttributeError, RuntimeError):
    print("TensorFlow deterministic-operation control is unavailable.")


def make_early_stopping():
    cfg = BASELINE_HYPERPARAMETERS["early_stopping"]
    return EarlyStopping(
        monitor=cfg["monitor"],
        patience=cfg["patience"],
        min_delta=cfg["min_delta"],
        restore_best_weights=cfg["restore_best_weights"],
        verbose=1,
    )


def benchmark_full_test_prediction(model, X_input):
    """3 warm-ups followed by 10 measured full-test predictions."""
    warmup_size = min(1024, len(X_input))
    for _ in range(N_WARMUP_RUNS):
        model.predict(X_input[:warmup_size], verbose=0)

    measured_times = []
    probabilities = None
    for _ in range(N_BENCHMARK_RUNS):
        start_time = time.perf_counter()
        probabilities = model.predict(X_input, verbose=0)
        measured_times.append(time.perf_counter() - start_time)

    mean_inference_time = float(np.mean(measured_times))
    return probabilities, {
        "Inference_Time_s": mean_inference_time,
        "Latency_ms_per_sample": mean_inference_time / len(X_input) * 1000.0,
        "Throughput_samples_per_s": len(X_input) / mean_inference_time,
    }


def calculate_fpr_metrics(y_true, y_pred, labels):
    matrices = multilabel_confusion_matrix(y_true, y_pred, labels=labels)
    per_class_fpr, false_positives, true_negatives = [], [], []

    for matrix in matrices:
        tn, fp, fn, tp = matrix.ravel()
        denominator = fp + tn
        per_class_fpr.append(fp / denominator if denominator else 0.0)
        false_positives.append(fp)
        true_negatives.append(tn)

    per_class_fpr = np.asarray(per_class_fpr, dtype=float)
    false_positives = np.asarray(false_positives, dtype=float)
    true_negatives = np.asarray(true_negatives, dtype=float)
    supports = np.asarray([(y_true == label).sum() for label in labels], dtype=float)
    overall_denominator = false_positives.sum() + true_negatives.sum()

    return {
        "FPR_macro": float(per_class_fpr.mean()),
        "FPR_micro_overall": float(
            false_positives.sum() / overall_denominator
            if overall_denominator else 0.0
        ),
        "FPR_weighted": float(np.average(per_class_fpr, weights=supports)),
    }, per_class_fpr, matrices


def calculate_test_metrics(y_true, y_pred, probabilities, labels):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "Recall_macro": recall_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "F1_macro": f1_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "Precision_weighted": precision_score(
            y_true, y_pred, labels=labels, average="weighted", zero_division=0
        ),
        "Recall_weighted": recall_score(
            y_true, y_pred, labels=labels, average="weighted", zero_division=0
        ),
        "F1_weighted": f1_score(
            y_true, y_pred, labels=labels, average="weighted", zero_division=0
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

    try:
        metrics["ROC_AUC_macro_OvR"] = roc_auc_score(
            y_true,
            probabilities,
            labels=labels,
            average="macro",
            multi_class="ovr",
        )
    except ValueError as error:
        metrics["ROC_AUC_macro_OvR"] = np.nan
        print("Macro ROC-AUC could not be calculated:", error)

    fpr_summary, per_class_fpr, class_matrices = calculate_fpr_metrics(
        y_true, y_pred, labels
    )
    metrics.update(fpr_summary)
    return metrics, per_class_fpr, class_matrices


def format_mean_sd(mean_value, sd_value, decimals=6, percentage=False):
    if pd.isna(mean_value):
        return "NA"
    multiplier = 100.0 if percentage else 1.0
    mean_value *= multiplier
    sd_value *= multiplier
    if pd.isna(sd_value):
        return f"{mean_value:.{decimals}f} ± NA"
    return f"{mean_value:.{decimals}f} ± {sd_value:.{decimals}f}"


def save_confusion_matrix_outputs(y_true, y_pred, output_stem):
    matrix = confusion_matrix(
        y_true, y_pred, labels=np.arange(number_of_classes)
    )
    matrix_df = pd.DataFrame(
        matrix,
        index=label_encoder.classes_,
        columns=label_encoder.classes_,
    )
    matrix_df.index.name = "True_Label"
    matrix_df.columns.name = "Predicted_Label"
    matrix_df.to_csv(f"{output_stem}.csv")

    # Publication-style confusion matrix consistent with the other experiments.
    plt.figure(figsize=(6.8, 5.5))

    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )

    plt.xticks(
        rotation=45,
        ha="right",
        fontsize=10,
    )
    plt.yticks(
        rotation=0,
        fontsize=10,
    )
    plt.xlabel(
        "Predicted Label",
        fontsize=11,
    )
    plt.ylabel(
        "True Label",
        fontsize=11,
    )
    plt.tight_layout()

    plt.savefig(
        f"{output_stem}.pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"{output_stem}.png",
        dpi=600,
        bbox_inches="tight",
    )
    plt.close()

def build_per_class_table(y_true, y_pred, labels, per_class_fpr, class_matrices):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )
    rows = []
    for idx, label in enumerate(labels):
        tn, fp, fn, tp = class_matrices[idx].ravel()
        rows.append({
            "Class_Code": int(label),
            "Class": str(label_encoder.classes_[label]),
            "Precision": float(precision[idx]),
            "Recall": float(recall[idx]),
            "F1": float(f1[idx]),
            "FPR": float(per_class_fpr[idx]),
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "Support": int(support[idx]),
        })
    return pd.DataFrame(rows)


TensorFlow deterministic operations enabled.


## 5. Fixed DNN baseline architecture


In [12]:
num_features = X_train_scaled.shape[1]
X_train_input = X_train_scaled
X_val_input = X_val_scaled
X_test_input = X_test_scaled


def build_baseline_model():
    p = BASELINE_HYPERPARAMETERS
    reg = regularizers.l2(p["l2_regularization"])
    model = Sequential(name="Baseline_DNN")
    model.add(Input(shape=(num_features,)))
    model.add(Dense(p["hidden_units_1"], activation="relu", kernel_regularizer=reg))
    model.add(Dropout(p["dropout_rate"]))
    model.add(Dense(p["hidden_units_2"], activation="relu", kernel_regularizer=reg))
    model.add(Dropout(p["dropout_rate"]))
    model.add(Dense(number_of_classes, activation="softmax", kernel_regularizer=reg))
    model.compile(
        optimizer=Adam(learning_rate=p["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

print("DNN input shapes:", X_train_input.shape, X_val_input.shape, X_test_input.shape)


DNN input shapes: (106878, 40) (14826, 40) (29433, 40)


## 6. Five-seed DNN training and evaluation


In [13]:
# ============================================================
# Five independent final-training runs: seeds 42-46
# Data split, SMOTE output, scaler, architecture, and settings stay fixed.
# ============================================================
MODEL_NAME = "DNN"
OUTPUT_DIRECTORY = Path.cwd() / "Baseline_Balanced_Without_FA_NSL_KDD_DNN_5_seed_results"
ARTIFACT_DIRECTORY = OUTPUT_DIRECTORY / "reusable_artifacts"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Save reusable fixed preprocessing/configuration once.
joblib.dump(classifier_scaler, ARTIFACT_DIRECTORY / "standard_scaler.joblib")
joblib.dump(label_encoder, ARTIFACT_DIRECTORY / "label_encoder.joblib")
with (ARTIFACT_DIRECTORY / "feature_columns.json").open("w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)
with (ARTIFACT_DIRECTORY / "smote_sampling_strategy.json").open("w", encoding="utf-8") as f:
    json.dump({str(k): int(v) for k, v in target_class_counts.items()}, f, indent=2)
np.savez_compressed(
    ARTIFACT_DIRECTORY / "fixed_split_indices.npz",
    train=train_indices,
    validation=validation_indices,
    test=test_indices,
)

experiment_config = {
    "Experiment": "Baseline Balanced NSL-KDD without feature augmentation and without Optuna",
    "Model": MODEL_NAME,
    "Dataset_Path": str(DATASET_PATH),
    "Split_Random_State": int(SEED),
    "Split": {"Train": 0.702, "Validation": 0.298 * (1.0 - 0.665), "Test": 0.298 * 0.665},
    "SMOTE_Random_State": int(SEED),
    "SMOTE_Target_Class_Counts": {str(k): int(v) for k, v in target_class_counts.items()},
    "Run_Seeds": RUN_SEEDS,
    "Hyperparameters": BASELINE_HYPERPARAMETERS,
    "Inference_Protocol": {
        "Warmup_Runs": N_WARMUP_RUNS,
        "Measured_Runs": N_BENCHMARK_RUNS,
        "Timer": "time.perf_counter",
    },
}
with (ARTIFACT_DIRECTORY / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_config, f, indent=2)

labels = np.arange(number_of_classes)
run_rows = []
per_class_rows = []
representative_candidates = []

for run_number, seed in enumerate(RUN_SEEDS, start=1):
    print(f"\n{'='*18} {MODEL_NAME} run {run_number}/5 — seed {seed} {'='*18}")

    seed_directory = OUTPUT_DIRECTORY / f"{MODEL_NAME}_Seed_{run_number}"
    seed_directory.mkdir(parents=True, exist_ok=True)

    tf.keras.backend.clear_session()
    set_final_run_seed(seed)
    model = build_baseline_model()
    early_stop = make_early_stopping()

    start_training = time.perf_counter()
    history = model.fit(
        X_train_input,
        y_train_resampled,
        validation_data=(X_val_input, y_val),
        epochs=BASELINE_HYPERPARAMETERS["epochs"],
        batch_size=BASELINE_HYPERPARAMETERS["batch_size"],
        shuffle=BASELINE_HYPERPARAMETERS["shuffle"],
        callbacks=[early_stop],
        verbose=1,
    )
    training_time = time.perf_counter() - start_training

    # Validation macro F1 for run diagnostics only.
    val_probabilities = model.predict(X_val_input, verbose=0)
    val_predictions = np.argmax(val_probabilities, axis=1)
    validation_macro_f1 = f1_score(
        y_val, val_predictions, average="macro", zero_division=0
    )

    # Fixed timing protocol also returns final full-test probabilities.
    test_probabilities, timing_metrics = benchmark_full_test_prediction(
        model, X_test_input
    )
    test_predictions = np.argmax(test_probabilities, axis=1)

    test_loss = float(model.evaluate(X_test_input, y_test, verbose=0)[0])
    test_metrics, per_class_fpr, class_matrices = calculate_test_metrics(
        y_test, test_predictions, test_probabilities, labels
    )
    test_metrics.update(timing_metrics)
    test_metrics["Training_Time_s"] = float(training_time)
    test_metrics["Test_Loss"] = test_loss
    test_metrics["Validation_F1_macro"] = float(validation_macro_f1)
    test_metrics["Epochs_Completed"] = int(len(history.history.get("loss", [])))

    run_row = {"Model": MODEL_NAME, "Run": int(run_number), "Seed": int(seed)}
    run_row.update(test_metrics)
    run_rows.append(run_row)

    # Save all scalar metrics and readable macro/weighted subsets per seed.
    pd.DataFrame([run_row]).to_csv(seed_directory / "scalar_metrics.csv", index=False)

    macro_columns = [
        "Model", "Run", "Seed", "Precision_macro", "Recall_macro", "F1_macro",
        "ROC_AUC_macro_OvR", "FPR_macro"
    ]
    weighted_columns = [
        "Model", "Run", "Seed", "Precision_weighted", "Recall_weighted",
        "F1_weighted", "FPR_weighted"
    ]
    pd.DataFrame([run_row])[macro_columns].to_csv(
        seed_directory / "macro_metrics.csv", index=False
    )
    pd.DataFrame([run_row])[weighted_columns].to_csv(
        seed_directory / "weighted_average_metrics.csv", index=False
    )

    report = classification_report(
        y_test,
        test_predictions,
        labels=labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(
        seed_directory / "classification_report.csv"
    )

    per_class_df = build_per_class_table(
        y_test, test_predictions, labels, per_class_fpr, class_matrices
    )
    per_class_df.insert(0, "Seed", int(seed))
    per_class_df.insert(0, "Run", int(run_number))
    per_class_df.insert(0, "Model", MODEL_NAME)
    per_class_df.to_csv(seed_directory / "per_class_metrics_and_fpr.csv", index=False)
    per_class_rows.extend(per_class_df.to_dict(orient="records"))

    pd.DataFrame([{
        "Model": MODEL_NAME,
        "Run": int(run_number),
        "Seed": int(seed),
        "FPR_macro": test_metrics["FPR_macro"],
        "FPR_micro_overall": test_metrics["FPR_micro_overall"],
        "FPR_weighted": test_metrics["FPR_weighted"],
    }]).to_csv(seed_directory / "false_positive_rate_summary.csv", index=False)

    save_confusion_matrix_outputs(
        y_test,
        test_predictions,
        seed_directory / "confusion_matrix",
    )

    pd.DataFrame(history.history).to_csv(
        seed_directory / "training_history.csv", index=False
    )

    prediction_df = pd.DataFrame({
        "True_Code": y_test,
        "True_Label": label_encoder.inverse_transform(y_test),
        "Predicted_Code": test_predictions,
        "Predicted_Label": label_encoder.inverse_transform(test_predictions),
    })
    for class_code, class_name in enumerate(label_encoder.classes_):
        prediction_df[f"Probability_{class_name}"] = test_probabilities[:, class_code]
    prediction_df.to_csv(
        seed_directory / "test_predictions_and_probabilities.csv", index=False
    )

    model_path = seed_directory / f"{MODEL_NAME.lower()}_seed_{seed}.keras"
    model.save(model_path)

    run_metadata = {
        "Model": MODEL_NAME,
        "Run": int(run_number),
        "Seed": int(seed),
        "Feature_Augmentation": False,
        "Optuna": False,
        "Training_Balancing": "SMOTE",
        "Split_Random_State": int(SEED),
        "SMOTE_Random_State": int(SEED),
        "Hyperparameters": BASELINE_HYPERPARAMETERS,
        "Training_Samples_After_SMOTE": int(len(X_train_input)),
        "Validation_Samples": int(len(X_val_input)),
        "Test_Samples": int(len(X_test_input)),
    }
    with (seed_directory / "run_metadata.json").open("w", encoding="utf-8") as f:
        json.dump(run_metadata, f, indent=2)

    representative_candidates.append({
        "Run": int(run_number),
        "Seed": int(seed),
        "F1_macro": float(test_metrics["F1_macro"]),
        "Predictions": test_predictions.copy(),
    })

    print(
        f"Completed seed {seed}: validation macro F1={validation_macro_f1:.6f}, "
        f"test macro F1={test_metrics['F1_macro']:.6f}"
    )

    del history, model
    gc.collect()
    tf.keras.backend.clear_session()

all_run_results = pd.DataFrame(run_rows)
all_per_class_results = pd.DataFrame(per_class_rows)

all_run_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_5_seed_run_results.csv", index=False
)
all_per_class_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_per_class_5_seed_run_results.csv", index=False
)

macro_metric_columns = [
    "Model", "Run", "Seed", "Precision_macro", "Recall_macro", "F1_macro",
    "ROC_AUC_macro_OvR", "FPR_macro"
]
weighted_metric_columns = [
    "Model", "Run", "Seed", "Precision_weighted", "Recall_weighted",
    "F1_weighted", "FPR_weighted"
]

macro_metrics_5_seed_results = all_run_results[macro_metric_columns].copy()
weighted_average_metrics_5_seed_results = all_run_results[weighted_metric_columns].copy()

macro_metrics_5_seed_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_macro_metrics_5_seed_results.csv", index=False
)
weighted_average_metrics_5_seed_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_weighted_average_metrics_5_seed_results.csv", index=False
)

display(all_run_results)
display(macro_metrics_5_seed_results)
display(weighted_average_metrics_5_seed_results)



================== DNN run 1/5 — seed 42 ==================
Epoch 1/100


2026-10-01 12:35:05.894605: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-10-01 12:35:06.170155: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


1658/1670 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8894 - loss: 0.3771

2026-10-01 12:35:12.076889: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


1670/1670 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9296 - loss: 0.2350 - val_accuracy: 0.9709 - val_loss: 0.1085
Epoch 2/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9598 - loss: 0.1320 - val_accuracy: 0.9767 - val_loss: 0.0928
Epoch 3/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9664 - loss: 0.1154 - val_accuracy: 0.9797 - val_loss: 0.0844
Epoch 4/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9696 - loss: 0.1061 - val_accuracy: 0.9812 - val_loss: 0.0811
Epoch 5/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9732 - loss: 0.0994 - val_accuracy: 0.9823 - val_loss: 0.0771
Epoch 6/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9741 - loss: 0.0952 - val_accuracy: 0.9822 - val_loss: 0.0745
Epoch 7/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9754 - loss: 0.0913 - val_accuracy: 0.9839 - val_loss: 0.0752
Epoch 8/100
1670/1670 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9763 - loss: 0.0898 - val_

...

2026-10-01 12:53:57.946924: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-01 12:54:03.271721: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Completed seed 46: validation macro F1=0.863936, test macro F1=0.850081


,Model,Run,Seed,Accuracy,Precision_macro,Recall_macro,F1_macro,Precision_weighted,Recall_weighted,F1_weighted,...,FPR_macro,FPR_micro_overall,FPR_weighted,Inference_Time_s,Latency_ms_per_sample,Throughput_samples_per_s,Training_Time_s,Test_Loss,Validation_F1_macro,Epochs_Completed
0,DNN,1,42,0.987123,0.842058,0.887459,0.857113,0.987423,0.987123,0.987226,...,0.004042,0.003219,0.007334,1.360389,0.046220,21635.723439,245.413334,0.063341,0.869239,54
1,DNN,2,43,0.986580,0.821081,0.874377,0.833005,0.987320,0.986580,0.986840,...,0.004223,0.003355,0.007693,1.401767,0.047626,20997.071954,239.573848,0.063549,0.861090,52
2,DNN,3,44,0.986919,0.842161,0.908447,0.866088,0.987372,0.986919,0.987100,...,0.003869,0.003270,0.006262,1.332884,0.045285,22082.190380,165.126226,0.064383,0.882585,36
3,DNN,4,45,0.986546,0.828680,0.891670,0.844283,0.987179,0.986546,0.986757,...,0.004250,0.003364,0.007794,1.364567,0.046362,21569.485474,169.344809,0.064413,0.860445,37
4,DNN,5,46,0.987599,0.830159,0.899286,0.850081,0.988268,0.987599,0.987864,...,0.003779,0.003100,0.006493,1.346975,0.045764,21851.177163,263.708712,0.062970,0.863936,57


,Model,Run,Seed,Precision_macro,Recall_macro,F1_macro,ROC_AUC_macro_OvR,FPR_macro
0,DNN,1,42,0.842058,0.887459,0.857113,0.997464,0.004042
1,DNN,2,43,0.821081,0.874377,0.833005,0.996592,0.004223
2,DNN,3,44,0.842161,0.908447,0.866088,0.997587,0.003869
3,DNN,4,45,0.828680,0.891670,0.844283,0.997215,0.004250
4,DNN,5,46,0.830159,0.899286,0.850081,0.996878,0.003779


,Model,Run,Seed,Precision_weighted,Recall_weighted,F1_weighted,FPR_weighted
0,DNN,1,42,0.987423,0.987123,0.987226,0.007334
1,DNN,2,43,0.987320,0.986580,0.986840,0.007693
2,DNN,3,44,0.987372,0.986919,0.987100,0.006262
3,DNN,4,45,0.987179,0.986546,0.986757,0.007794
4,DNN,5,46,0.988268,0.987599,0.987864,0.006493


## 7. Five-run mean ± sample SD and representative DNN run


In [15]:
# ============================================================
# Five-run mean ± sample SD summaries
# ============================================================
identifier_columns = {"Model", "Run", "Seed"}
metric_columns = [c for c in all_run_results.columns if c not in identifier_columns]
summary_records = []

for metric_name in metric_columns:
    values = pd.to_numeric(all_run_results[metric_name], errors="coerce")
    metric_mean = float(values.mean())
    metric_sd = float(values.std(ddof=1))
    summary_records.append({
        "Model": "DNN",
        "Metric": metric_name,
        "Mean": metric_mean,
        "SD": metric_sd,
        "N": int(values.notna().sum()),
        "Mean ± SD": format_mean_sd(metric_mean, metric_sd),
    })

five_run_summary_long = pd.DataFrame(summary_records)
five_run_summary_table = five_run_summary_long.pivot(
    index="Model", columns="Metric", values="Mean ± SD"
).reindex(["DNN"])

five_run_summary_long.to_csv(
    OUTPUT_DIRECTORY / "five_run_mean_sd_long.csv", index=False
)
five_run_summary_table.to_csv(
    OUTPUT_DIRECTORY / "five_run_mean_sd_report_table.csv"
)

macro_summary_names = [
    "Precision_macro", "Recall_macro", "F1_macro", "ROC_AUC_macro_OvR", "FPR_macro"
]
weighted_summary_names = [
    "Precision_weighted", "Recall_weighted", "F1_weighted", "FPR_weighted"
]

macro_metrics_mean_sd = five_run_summary_long[
    five_run_summary_long["Metric"].isin(macro_summary_names)
].reset_index(drop=True)
weighted_average_metrics_mean_sd = five_run_summary_long[
    five_run_summary_long["Metric"].isin(weighted_summary_names)
].reset_index(drop=True)

macro_metrics_mean_sd.to_csv(
    OUTPUT_DIRECTORY / "macro_metrics_5_seed_mean_sd.csv", index=False
)
weighted_average_metrics_mean_sd.to_csv(
    OUTPUT_DIRECTORY / "weighted_average_metrics_5_seed_mean_sd.csv", index=False
)

# Per-class five-run mean ± sample SD.
per_class_summary_records = []
for class_name, class_rows in all_per_class_results.groupby("Class", sort=False):
    record = {
        "Model": "DNN",
        "Class": class_name,
        "Support": int(class_rows["Support"].iloc[0]),
        "N": int(len(class_rows)),
    }
    for metric_name in ["Precision", "Recall", "F1", "FPR"]:
        metric_mean = float(class_rows[metric_name].mean())
        metric_sd = float(class_rows[metric_name].std(ddof=1))
        record[f"{metric_name}_Mean"] = metric_mean
        record[f"{metric_name}_SD"] = metric_sd
        record[f"{metric_name}_Mean ± SD"] = format_mean_sd(metric_mean, metric_sd)
    per_class_summary_records.append(record)

per_class_mean_sd = pd.DataFrame(per_class_summary_records)
per_class_mean_sd.to_csv(
    OUTPUT_DIRECTORY / "per_class_five_run_mean_sd.csv", index=False
)

# Representative run: test macro F1 closest to the five-run mean.
mean_macro_f1 = float(all_run_results["F1_macro"].mean())
representative = min(
    representative_candidates,
    key=lambda item: abs(item["F1_macro"] - mean_macro_f1),
)
representative_runs = pd.DataFrame([{
    "Model": "DNN",
    "Representative_Run": representative["Run"],
    "Representative_Seed": representative["Seed"],
    "Representative_F1_macro": representative["F1_macro"],
    "Five_Run_Mean_F1_macro": mean_macro_f1,
}])
representative_runs.to_csv(
    OUTPUT_DIRECTORY / "representative_runs.csv", index=False
)

save_confusion_matrix_outputs(
    y_test,
    representative["Predictions"],
    OUTPUT_DIRECTORY / "dnn_representative_confusion_matrix",
)
representative_report = classification_report(
    y_test,
    representative["Predictions"],
    labels=np.arange(number_of_classes),
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(representative_report).transpose().to_csv(
    OUTPUT_DIRECTORY / "dnn_representative_classification_report.csv"
)

# Compact macro and weighted journal tables in the preferred metric order.
def build_compact_journal_table(average_kind):
    summary_index = five_run_summary_long.set_index("Metric")
    if average_kind == "macro":
        mapping = [
            ("Accuracy", "Accuracy (%)", True, 2),
            ("Recall_macro", "Recall (macro %)", True, 2),
            ("Precision_macro", "Precision (macro %)", True, 2),
            ("F1_macro", "F1-score (macro %)", True, 2),
            ("FPR_macro", "FPR (macro %)", True, 4),
        ]
    else:
        mapping = [
            ("Accuracy", "Accuracy (%)", True, 2),
            ("Recall_weighted", "Recall (weighted %)", True, 2),
            ("Precision_weighted", "Precision (weighted %)", True, 2),
            ("F1_weighted", "F1-score (weighted %)", True, 2),
            ("FPR_weighted", "FPR (weighted %)", True, 4),
        ]

    row = {"Model": "DNN"}
    for metric, display_name, pct, decimals in mapping:
        row[display_name] = format_mean_sd(
            float(summary_index.loc[metric, "Mean"]),
            float(summary_index.loc[metric, "SD"]),
            decimals=decimals,
            percentage=pct,
        )
    return pd.DataFrame([row])

macro_journal_table = build_compact_journal_table("macro")
weighted_journal_table = build_compact_journal_table("weighted")
macro_journal_table.to_csv(
    OUTPUT_DIRECTORY / "journal_macro_mean_sd_table.csv", index=False
)
weighted_journal_table.to_csv(
    OUTPUT_DIRECTORY / "journal_weighted_mean_sd_table.csv", index=False
)

print("Five-run mean ± sample SD (n=5):")
display(five_run_summary_table)
print("Macro mean ± SD:")
display(macro_metrics_mean_sd)
print("Weighted mean ± SD:")
display(weighted_average_metrics_mean_sd)
print("Representative run:")
display(representative_runs)
print("All outputs saved to:", OUTPUT_DIRECTORY.resolve())


Five-run mean ± sample SD (n=5):


Metric,Accuracy,Epochs_Completed,F1_macro,F1_weighted,FPR_macro,FPR_micro_overall,FPR_weighted,Inference_Time_s,Latency_ms_per_sample,MCC,Precision_macro,Precision_weighted,ROC_AUC_macro_OvR,Recall_macro,Recall_weighted,Test_Loss,Throughput_samples_per_s,Training_Time_s,Validation_F1_macro
Model,,,,,,,,,,,,,,,,,,,
DNN,0.986953 ± 0.000434,47.200000 ± 9.934787,0.850114 ± 0.012559,0.987157 ± 0.000438,0.004032 ± 0.000209,0.003262 ± 0.000108,0.007115 ± 0.000700,1.361316 ± 0.025796,0.046251 ± 0.000876,0.977975 ± 0.000739,0.832828 ± 0.009146,0.987512 ± 0.000432,0.997147 ± 0.000412,0.892248 ± 0.012792,0.986953 ± 0.000434,0.063731 ± 0.000643,21627.129682 ± 405.477176,216.633386 ± 45.988677,0.867459 ± 0.009140


Macro mean ± SD:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Precision_macro,0.832828,0.009146,5,0.832828 ± 0.009146
1,DNN,Recall_macro,0.892248,0.012792,5,0.892248 ± 0.012792
2,DNN,F1_macro,0.850114,0.012559,5,0.850114 ± 0.012559
3,DNN,ROC_AUC_macro_OvR,0.997147,0.000412,5,0.997147 ± 0.000412
4,DNN,FPR_macro,0.004032,0.000209,5,0.004032 ± 0.000209


Weighted mean ± SD:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Precision_weighted,0.987512,0.000432,5,0.987512 ± 0.000432
1,DNN,Recall_weighted,0.986953,0.000434,5,0.986953 ± 0.000434
2,DNN,F1_weighted,0.987157,0.000438,5,0.987157 ± 0.000438
3,DNN,FPR_weighted,0.007115,0.000700,5,0.007115 ± 0.000700


Representative run:


,Model,Representative_Run,Representative_Seed,Representative_F1_macro,Five_Run_Mean_F1_macro
0,DNN,5,46,0.850081,0.850114


All outputs saved to: /kaggle/working/Baseline_Balanced_Without_FA_NSL_KDD_DNN_5_seed_results
